# 🚀 ColabAI Studio - Máy Chủ AI Đám Mây với RAG & Xác Thực Bảo Mật

Notebook này đóng vai trò là **Backend AI RAG** trên GPU Google Colab miễn phí.

### ⚙️ Các Tính Năng Được Tích Hợp:
1. **Bảo mật Đăng nhập:** Kiểm tra tài khoản & mật khẩu trước khi cho phép Client máy tính truy cập.
2. **Hệ thống RAG (Retrieval-Augmented Generation):**
   - Cào tài liệu từ URL
   - Chia nhỏ thành các chunks (Chunking)
   - Sinh vector embeddings bằng `sentence-transformers`
   - Tìm kiếm vector top-k đoạn văn bản liên quan nhất tới câu hỏi
   - Ghép ngữ cảnh và đưa vào LLM (`Qwen2.5-Instruct`) sinh câu trả lời có trích dẫn nguồn.
3. **Cloudflare Tunnel:** Mở đường hầm HTTPS công khai miễn phí 100% không giới hạn port.

### 📌 Hướng Dẫn Chạy:
1. Menu **Runtime** &rarr; **Change runtime type** &rarr; Chọn **T4 GPU** &rarr; Bấm **Save**.
2. Menu **Runtime** &rarr; **Run all**.
3. Lấy URL và tài khoản ở ô cuối cùng để dán vào màn hình đăng nhập máy tính của bạn.

In [ ]:
# BƯỚC 1: Cài đặt các thư viện cần thiết
!pip install -q fastapi uvicorn requests beautifulsoup4 transformers accelerate torch sentence-transformers nest-asyncio

In [ ]:
# BƯỚC 2: Tải mô hình Embedding RAG & Mô hình LLM
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"🎮 Thiết bị đang sử dụng: {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"⚡ GPU: {torch.cuda.get_device_name(0)}")

print("⏳ [1/2] Đang tải mô hình Embedding RAG...")
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

print(f"⏳ [2/2] Đang tải mô hình LLM ({LLM_MODEL})...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Toàn bộ mô hình RAG và LLM đã sẵn sàng!")

In [ ]:
# BƯỚC 3: Cấu hình Tài khoản, RAG Engine & FastAPI Backend
import time, re, subprocess, requests, uuid
import numpy as np
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException, Header, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List
import uvicorn, nest_asyncio

nest_asyncio.apply()

# TÀI KHOẢN COLAB (Bạn có thể đổi nếu muốn)
COLAB_USER = "admin"
COLAB_PASS = "colab2026"
ACTIVE_TOKENS = set()

app = FastAPI(title="ColabAI RAG")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Hàm cào dữ liệu URL
def scrape_url(url: str):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}
    resp = requests.get(url, headers=headers, timeout=12)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    title = soup.title.string.strip() if soup.title and soup.title.string else url
    paragraphs = []
    for p in soup.find_all(["p", "h1", "h2", "h3", "li"]):
        text = p.get_text(separator=" ", strip=True)
        if len(text) > 30:
            paragraphs.append(text)
    clean = "\n\n".join(paragraphs)
    return {"title": title, "content": clean, "word_count": len(clean.split())}

# Hàm Chunking & RAG Vector Retrieval
def chunk_text(text: str, chunk_size: int = 250, chunk_overlap: int = 40):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        ch = " ".join(words[start:end])
        if ch.strip(): chunks.append(ch)
        start += (chunk_size - chunk_overlap)
        if start >= len(words): break
    return chunks

def search_rag_chunks(query: str, text: str, top_k: int = 3):
    chunks = chunk_text(text)
    if not chunks: return []
    chunk_embs = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
    query_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = np.dot(query_emb, chunk_embs.T)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{"chunk_id": int(i+1), "text": chunks[i], "score": round(float(scores[i]), 4)} for i in top_idx]

# Auth check
class LoginReq(BaseModel):
    username: str
    password: str

class ProcessReq(BaseModel):
    url: str
    prompt: str = "Tóm tắt nội dung"
    mode: str = "summary"
    use_rag: bool = True
    top_k: int = 3

@app.post("/api/auth/login")
def login(req: LoginReq):
    if req.username == COLAB_USER and req.password == COLAB_PASS:
        tok = f"colab_{uuid.uuid4().hex[:16]}"
        ACTIVE_TOKENS.add(tok)
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        return {"status": "authenticated", "token": tok, "username": req.username, "gpu": gpu}
    raise HTTPException(status_code=401, detail="Sai tên đăng nhập hoặc mật khẩu Colab! Mặc định: admin / colab2026")

@app.get("/health")
def health():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "online", "gpu": f"{gpu} Active", "rag": True, "time": time.time()}

@app.post("/api/process")
def process(req: ProcessReq, authorization: Optional[str] = Header(None)):
    if not authorization or authorization.replace("Bearer ", "").strip() not in ACTIVE_TOKENS:
        raise HTTPException(status_code=401, detail="Chưa xác thực hoặc token hết hạn. Vui lòng đăng nhập lại.")
    
    t0 = time.time()
    try:
        scraped = scrape_url(req.url)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Lỗi cào URL: {str(e)}")

    # RAG Retrieval
    rag_sources = search_rag_chunks(req.prompt, scraped["content"], top_k=req.top_k) if req.use_rag else []
    ctx = "\n\n".join([f"[Đoạn #{s['chunk_id']} - Điểm liên quan: {int(s['score']*100)}%]:\n{s['text']}" for s in rag_sources]) if rag_sources else scraped["content"][:3500]

    system_msg = "Bạn là chuyên gia phân tích dữ liệu AI RAG. Hãy dựa vào các đoạn trích dẫn được cung cấp từ tài liệu để trả lời câu hỏi của người dùng bằng tiếng Việt chi tiết, mạch lạc theo định dạng Markdown:"
    user_msg = f"TIÊU ĐỀ: {scraped['title']}\nURL: {req.url}\n\nCÁC ĐOẠN TRÍCH DẪN RAG:\n{ctx}\n\nYÊU CẦU: {req.prompt}\nCHẾ ĐỘ: {req.mode}\nHãy trả lời chi tiết và nêu rõ thông tin căn cứ từ các đoạn trích:"

    inputs = tokenizer([tokenizer.apply_chat_template([{"role":"system","content":system_msg},{"role":"user","content":user_msg}], tokenize=False, add_generation_prompt=True)], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=1024, temperature=0.6, top_p=0.9, repetition_penalty=1.1, do_sample=True)
        answer = tokenizer.batch_decode([out[len(inp):] for inp, out in zip(inputs.input_ids, gen_ids)], skip_special_tokens=True)[0]

    return {
        "status": "success",
        "result": answer,
        "rag_sources": rag_sources,
        "metadata": {
            "url": req.url,
            "title": scraped["title"],
            "word_count": scraped["word_count"],
            "retrieved_chunks": len(rag_sources),
            "inference_time_sec": round(time.time() - t0, 2),
            "model": LLM_MODEL,
            "embedder": EMBED_MODEL
        }
    }
print("✅ FastAPI Backend RAG đã cấu hình xong!")

In [ ]:
# BƯỚC 4: Tạo Cloudflare Tunnel & Khởi Chạy (LẤY LINK & MẬT KHẨU Ở ĐÂY)
import os, subprocess, re

if not os.path.exists("./cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("=" * 65)
print("🎉 MÁY CHỦ COLAB RAG ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
print("👉 DÁN LINK NÀY VÀO MÀN HÌNH ĐĂNG NHẬP MÁY TÍNH:")
print(f"\n     {public_url}\n")
print(f"🔑 TÀI KHOẢN ĐĂNG NHẬP COLAB:")
print(f"   - Tên đăng nhập: {COLAB_USER}")
print(f"   - Mật khẩu:      {COLAB_PASS}")
print("=" * 65)

uvicorn.run(app, host="127.0.0.1", port=8000)